Notebook 4: Tomographyic-like approach
===============================================================================

In [1]:
# import libraries
import sys
import copy
import numpy as np
import pickle

sys.path.insert(1, '../swa')

from utils import *
from _stream import SeismicStream
from _curve import Curve

import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Qt5Agg')

In [3]:
# Define the directories
method = 'fdbf'
version = 'manual'
prj_dir = '../data/syn_data'

path2raw = os.path.join(prj_dir,'raw')
path2geom = f'{prj_dir}/geometry_test.csv' # Directory containing the geometry file
path2fk = f'{prj_dir}/proc/fk_filter' # Directory containing the FK filter file
path2diffs = f'{prj_dir}/proc/phase_diffs'

path2plot = 'figures/'
ext = '.sgy'  # Shot file extension
#filtname = 'fkfilter_synth_test.txt' # fk filter file
safe_makedirs(path2plot)

# List of directories to create
directories = [path2raw, path2geom, path2fk, path2diffs, path2plot]
def create_dir(dir_list):
    for directory in dir_list:
        if not os.path.exists(directory):
            os.makedirs(directory)
            print(f"📁 Directory created: {directory}")
create_dir(directories)


In [4]:
# processing and plotting settings
settings = create_settings_dict(trafo = method,             # transformation type
                         zero_padding=False, freq_step=0.5,  # zero padding
                         normalize = True,local_max = True, # amplitude normalization
                         picking = version,                # picking mode ("manual" or "auto")
                         fmin=2, fmax=80,                  # frequency range
                         vmin=50, vmax=1200, velstep=1)     # testing phase velocity range and step


# get paths to shot files
path2sht = [f.path for f in os.scandir(path2raw) if f.is_file()]
path2sht = natural_sort(path2sht)
path2sht = [[f] for f in path2sht]

## 1. Definizione dei range di calcolo

In [5]:
# define offset range for forward shots
min_offset_fw = 3
max_offset_fw = 1e6

# define offset range for reverse shots
min_offset_rw = -3
max_offset_rw = -1e6

# minimum receiver number
min_nrec = 6

## 2. Calcolo delle differenze di fase tra ricevitori adiacenti, per ogni frequenza

In [6]:
# %% phase differences
all_phase_diffs = []

for i in range(0,len(path2sht)):

    # read seismic data
    stream = SeismicStream(path2sht[i][0], settings) # record
    fids = stream._argfreq()                         # indices of frequency subset
    nrec = len(stream.receiver)

    # create copy of stream object
    stream_fw = copy.deepcopy(stream) # copy
    stream_rw = copy.deepcopy(stream)  # copy

    fn = os.path.splitext(os.path.basename(path2sht[i][0]))[0] # filename

    # %% process forward shots
    phase_diffs = np.empty((len(fids), nrec - 1))
    phase_diffs[:] = np.nan

    # define offset range
    offset = stream.receiver-stream.source              # signed offset vector
    omin = np.argmin(np.abs(offset - min_offset_fw))    # min offset id
    omax = np.argmin(np.abs(offset - max_offset_fw))    # max offset id
    oids = np.arange(omin, omax+1)                      # offset ids to be used

    if len(offset[oids]) >= min_nrec:

        # filter by offset (remove near offsets)
        stream_fw._trim_by_offsets(min_offset_fw, max_offset_fw)

        # apply fk filtering to remove higher modes
        fk_fname = f'{path2fk}/fw_shots/fkfilter_{fn}.txt'
        path, _ = os.path.split(fk_fname)
        safe_makedirs(path)

        # if file exits apply fk filter
        if os.path.isfile(fk_fname):
            stream_fw._fk_filter_from_file(fname=fk_fname,
                                        show=False)
        # otherwise pick in data
        else:
            stream_fw._fk_filter_from_pick(fname=fk_fname,
                                        show=False)

        # compute phase differences
        phase_diffs_sub, fids = stream_fw._compute_phasediffs()

        # store in array
        phase_diffs[:,oids[:-1]] = phase_diffs_sub

    # %% process reverse shots
    omax = np.argmin(np.abs(offset - min_offset_rw))
    omin = np.argmin(np.abs(offset - max_offset_rw))
    oids = np.arange(omin, omax+1)

    if len(offset[oids]) >= min_nrec:
        # filter by offset (remove near offsets)
        stream_rw._trim_by_offsets(min_offset_rw, max_offset_rw)

        # apply fk filtering to remove higher modes
        fk_fname = f'{path2fk}/rw_shots/fkfilter_{fn}.txt'
        path, _ = os.path.split(fk_fname)
        safe_makedirs(path)

        # if file exits apply fk filter
        if os.path.isfile(fk_fname):
            stream_rw._fk_filter_from_file(fname=fk_fname,
                                           show=False)
        # otherwise pick in data
        else:
            stream_rw._fk_filter_from_pick(fname=fk_fname,
                                           show=False)

        # compute phase differences
        phase_diffs_sub, fids = stream_rw._compute_phasediffs()

        # store in array
        phase_diffs[:,oids[:-1]] = phase_diffs_sub

    all_phase_diffs.append(phase_diffs)

# %% save all phase differences
safe_makedirs(path2diffs)
with open(f"{path2diffs}/all_phase_diffs.pickle", "wb") as fp:
    pickle.dump(all_phase_diffs, fp)
print('Observations saved in ', path2diffs)


KeyboardInterrupt



## 3. settings for tomography

In [7]:
# processing and plotting settings
settings = create_settings_dict(trafo = 'fdbf',             # transformation type
                         zero_padding=False, freq_step=0.5,  # zero padding
                         normalize = True,local_max = True, # amplitude normalization
                         picking = 'manual',                # picking mode ("manual" or "auto")
                         fmin=2, fmax=80,                  # frequency range
                         vmin=50, vmax=1200, velstep=1)     # testing phase velocity range and step


# get paths to shot files
path2sht = [f.path for f in os.scandir(path2raw) if f.is_file()]
path2sht = natural_sort(path2sht)
path2sht = [[f] for f in path2sht]

# file containing all phase differences per shots
with open(f"{path2diffs}/all_phase_diffs.pickle", "rb") as fp:
    all_phase_diffs = pickle.load(fp)

# define offset range for forward shots
min_offset_fw = 3
max_offset_fw = 1e6

# define offset range for reverse shots
min_offset_rw = -3
max_offset_rw = -1e6

# regularization parameter
lam = 20

# recording parameters
rec0 = SeismicStream(path2sht[0][0], settings)      # record0
nshots = len(path2sht)                              # number of shots
nrec = len(rec0.receiver)                           # number of receivers
dx = np.mean(np.diff(rec0.receiver))                # receiver separataion
dt = rec0.dt                                        # sampling rate in s
npts = rec0.npts                                    # number of samples
fids = rec0._argfreq()

# frequency range
freq = recordParam2freq(dt,npts)
freq_sel = freq[fids]

## 4. run tomo like for each fequency

In [8]:
# allocate space
phi_vel_all = np.zeros((nrec-1,len(freq_sel)))

for jj,f in enumerate(freq_sel):

    A = np.zeros((2*nshots*nrec-1,nrec-1)) # design matrix
    dphi = np.zeros((2*nshots*nrec-1)) # phase vector
    weights = np.ones((2*nshots*nrec-1)) # weight vector
    iii = 0

    for kk in range(0, len(path2sht)):

        stream = SeismicStream(path2sht[kk][0], settings)

        # %% process forward shots
        # define offset range
        offset = stream.receiver - stream.source        # signed offset vector
        omin = np.argmin(np.abs(offset - min_offset_fw))
        omax = np.argmin(np.abs(offset - max_offset_fw))
        oids = np.arange(omin, omax)

        # fill A and dphi
        for ii in oids:
            if all_phase_diffs[kk][jj,ii]<0:
                iii += 1
                A[iii,ii] = dx
                dphi[iii] = all_phase_diffs[kk][jj,ii]

        # %% process reverse shots
        # define offset range
        omin = np.argmin(np.abs(offset - min_offset_rw))
        omax = np.argmin(np.abs(offset - max_offset_rw))
        oids = np.arange(omin, omax)

        # fill A and dphi
        for ii in oids:
            if all_phase_diffs[kk][jj,ii]>0:
                iii += 1
                A[iii,ii] = dx
                dphi[iii] = all_phase_diffs[kk][jj,ii]

    # solve equations
    w = np.diag(weights)
    phi_vel, phi_model = tomo2D_phasediff(lam=lam, f=f, A=A, dphi=dphi, w=w)
    phi_vel_all[:,jj] = phi_vel

# save dispersion curves
for i in range(len(phi_vel_all)):
    dc = Curve()
    dc._init_data(freq_sel, phi_vel_all[i,:], err = None)
    dc._save(os.path.join(prj_dir,'dc'), f'dc{i}', format = 'csv')

## Plot

In [9]:
# %% show results
fig, ax = plt.subplots(3,1,figsize = (8.5,6), constrained_layout=True)

for kk in range(0, len(path2sht)):
    if kk == 0:
        label = f'f={round(freq_sel[15])} Hz'
    else:
        label = None
    ax[0].plot(range(nrec - 1), all_phase_diffs[kk][15,:], 'k.', label=label)

ax[0].set_ylabel(r'$\Delta \phi$ (rad)')
ax[0].set_xlabel('x (m)')
ax[0].set_xlim(rec0.receiver[0],rec0.receiver[-1])
ax[0].legend(loc='lower right', edgecolor = 'k', frameon = True)

ax[1].plot(range(nrec-1),phi_vel_all[:,0], 'k.-', label=f'f={round(freq_sel[15])} Hz')
ax[1].set_ylabel(r'$v_r$ (m/s)')
ax[1].set_xlabel('x (m)')
ax[1].set_xlim(rec0.receiver[0],rec0.receiver[-1])
ax[1].legend(loc='lower right', edgecolor = 'k', frameon = True)

im = ax[2].pcolor(range(nrec-1),freq_sel,phi_vel_all.T, vmin = 180, vmax = 550, cmap='turbo')
cbar = fig.colorbar(im, ax=ax[2], label = r'$v_r$ (m/s)')
ax[2].set_ylabel('f (Hz)')
ax[2].set_xlabel('x (m)')
ax[2].set_ylim(10, 40)
ax[2].set_title('Tomo 2D', fontsize=10)
#fig.savefig('../Rovereto/NB3_tomo2D.png', transparent=True, dpi=200)
plt.show()

In [11]:
# %% show results
fig, ax = plt.subplots(figsize = (8,4.8))
im = ax.pcolor(range(nrec-1),freq_sel,phi_vel_all.T, vmin = 180, vmax = 600, cmap='turbo')
cbar = fig.colorbar(im, ax=ax, label = r'$v_r$ (m/s)')
ax.set_ylabel('f (Hz)')
ax.set_xlabel('x (m)')
ax.set_ylim(5, 50)
ax.set_title('Tomo 2D', fontsize=10)
fig.savefig(os.path.join(path2plot,'4_tomo2D.png'), dpi=200)
plt.show()